<a href="https://colab.research.google.com/github/Technokemp/student-record-management-system-/blob/main/student_record_management_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Student Record Management System (Python + SQLite, runs in one Colab cell)

import sqlite3

DB_FILE = "student_records.db"


# ---------- Setup ----------

def connect():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON")
    conn.row_factory = sqlite3.Row
    return conn


def create_tables(conn):
    # Create core tables first
    conn.execute("""
        CREATE TABLE IF NOT EXISTS students (
            roll_no    TEXT PRIMARY KEY,
            name       TEXT NOT NULL,
            email      TEXT UNIQUE
        )""")

    # Automatically migrate schema: add dob column if it doesn't exist
    try:
        conn.execute("ALTER TABLE students ADD COLUMN dob TEXT")
    except sqlite3.OperationalError:
        # Column already exists, we can ignore
        pass

    conn.execute("""
        CREATE TABLE IF NOT EXISTS courses (
            course_code TEXT PRIMARY KEY,
            title       TEXT NOT NULL,
            credits     INTEGER NOT NULL CHECK (credits > 0)
        )""")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS enrollments (
            roll_no     TEXT NOT NULL REFERENCES students(roll_no) ON DELETE CASCADE,
            course_code TEXT NOT NULL REFERENCES courses(course_code) ON DELETE CASCADE,
            semester    TEXT NOT NULL,
            marks       REAL CHECK (marks IS NULL OR (marks >= 0 AND marks <= 100)),
            PRIMARY KEY (roll_no, course_code)
        )""")
    conn.commit()


def grade_for(marks):
    if marks is None:
        return "-"
    if marks >= 90:
        return "A+"
    if marks >= 80:
        return "A"
    if marks >= 70:
        return "B"
    if marks >= 60:
        return "C"
    if marks >= 50:
        return "D"
    return "F"


def ask(prompt, required=True):
    while True:
        value = input(prompt).strip()
        if value or not required:
            return value
        print("  This field is required.")


# ---------- Students ----------

def add_student(conn):
    roll = ask("Roll number: ")
    name = ask("Name: ")
    email = ask("Email (optional): ", required=False) or None
    dob = ask("Date of birth YYYY-MM-DD (optional): ", required=False) or None
    try:
        conn.execute(
            "INSERT INTO students (roll_no, name, email, dob) VALUES (?, ?, ?, ?)",
            (roll, name, email, dob),
        )
        conn.commit()
        print("Student added.")
    except sqlite3.IntegrityError as e:
        print("Could not add student:", e)


def list_students(conn):
    rows = conn.execute("SELECT roll_no, name, email, dob FROM students ORDER BY roll_no").fetchall()
    if not rows:
        print("No students found.")
        return
    print(f"\n{'Roll No':<15}{'Name':<25}{'Email':<30}{'DOB'}")
    print("-" * 80)
    for r in rows:
        print(f"{r['roll_no']:<15}{r['name']:<25}{(r['email'] or '-'):<30}{r['dob'] or '-'}")


def update_student(conn):
    roll = ask("Roll number of student to update: ")
    student = conn.execute("SELECT * FROM students WHERE roll_no = ?", (roll,)).fetchone()
    if not student:
        print("Student not found.")
        return
    print("Press Enter to keep the current value.")
    name = ask(f"Name [{student['name']}]: ", required=False) or student["name"]
    email = ask(f"Email [{student['email'] or '-'}]: ", required=False) or student["email"]
    dob = ask(f"DOB [{student['dob'] or '-'}]: ", required=False) or student["dob"]
    try:
        conn.execute(
            "UPDATE students SET name = ?, email = ?, dob = ? WHERE roll_no = ?",
            (name, email, dob, roll),
        )
        conn.commit()
        print("Student updated.")
    except sqlite3.IntegrityError as e:
        print("Could not update student:", e)


def delete_student(conn):
    roll = ask("Roll number of student to delete: ")
    confirm = ask(f"Delete {roll} and all their enrollments? (y/n): ").lower()
    if confirm != "y":
        print("Cancelled.")
        return
    cur = conn.execute("DELETE FROM students WHERE roll_no = ?", (roll,))
    conn.commit()
    print("Deleted." if cur.rowcount else "Student not found.")


def search_students(conn):
    term = ask("Search by roll number or name: ")
    rows = conn.execute(
        "SELECT roll_no, name, email FROM students WHERE roll_no LIKE ? OR name LIKE ?",
        (f"%{term}%", f"%{term}%"),
    ).fetchall()
    if not rows:
        print("No matches.")
    for r in rows:
        print(f"{r['roll_no']}  {r['name']}  {r['email'] or '-'}")


# ---------- Courses ----------

def add_course(conn):
    code = ask("Course code (e.g. CS301): ").upper()
    title = ask("Course title: ")
    try:
        credits = int(ask("Credits: "))
        conn.execute(
            "INSERT INTO courses (course_code, title, credits) VALUES (?, ?, ?)",
            (code, title, credits),
        )
        conn.commit()
        print("Course added.")
    except ValueError:
        print("Credits must be a whole number.")
    except sqlite3.IntegrityError as e:
        print("Could not add course:", e)


def list_courses(conn):
    rows = conn.execute("SELECT * FROM courses ORDER BY course_code").fetchall()
    if not rows:
        print("No courses found.")
        return
    print(f"\n{'Code':<12}{'Title':<35}{'Credits'}")
    print("-" * 55)
    for r in rows:
        print(f"{r['course_code']:<12}{r['title']:<35}{r['credits']}")


# ---------- Enrollments and marks ----------

def enroll_student(conn):
    roll = ask("Roll number: ")
    code = ask("Course code: ").upper()
    semester = ask("Semester (e.g. 2026-Odd): ")
    try:
        conn.execute(
            "INSERT INTO enrollments (roll_no, course_code, semester) VALUES (?, ?, ?)",
            (roll, code, semester),
        )
        conn.commit()
        print("Enrolled.")
    except sqlite3.IntegrityError as e:
        print("Could not enroll (check roll number and course code):", e)


def record_marks(conn):
    roll = ask("Roll number: ")
    code = ask("Course code: ").upper()
    try:
        marks = float(ask("Marks (0-100): "))
        cur = conn.execute(
            "UPDATE enrollments SET marks = ? WHERE roll_no = ? AND course_code = ?",
            (marks, roll, code),
        )
        conn.commit()
        print("Marks recorded." if cur.rowcount else "No such enrollment.")
    except ValueError:
        print("Marks must be a number.")
    except sqlite3.IntegrityError as e:
        print("Invalid marks:", e)


def transcript(conn):
    roll = ask("Roll number: ")
    student = conn.execute("SELECT name FROM students WHERE roll_no = ?", (roll,)).fetchone()
    if not student:
        print("Student not found.")
        return
    rows = conn.execute(
        """
        SELECT e.course_code, c.title, c.credits, e.semester, e.marks
        FROM enrollments e
        JOIN courses c ON c.course_code = e.course_code
        WHERE e.roll_no = ?
        ORDER BY e.semester, e.course_code
        """,
        (roll,),
    ).fetchall()

    print(f"\nTranscript for {student['name']} ({roll})")
    print(f"{'Semester':<12}{'Code':<10}{'Title':<30}{'Cr':<5}{'Marks':<8}{'Grade'}")
    print("-" * 75)
    grade_points = {"A+": 10, "A": 9, "B": 8, "C": 7, "D": 6, "F": 0}
    total_points, total_credits = 0.0, 0
    for r in rows:
        g = grade_for(r["marks"])
        marks_txt = "-" if r["marks"] is None else f"{r['marks']:.0f}"
        print(f"{r['semester']:<12}{r['course_code']:<10}{r['title'][:28]:<30}{r['credits']:<5}{marks_txt:<8}{g}")
        if g in grade_points:
            total_points += grade_points[g] * r["credits"]
            total_credits += r["credits"]
    if total_credits:
        print(f"\nWeighted average (graded courses): {total_points / total_credits:.2f} / 10")


# ---------- Menu ----------

MENU = {
    "1": ("Add student", add_student),
    "2": ("List students", list_students),
    "3": ("Update student", update_student),
    "4": ("Delete student", delete_student),
    "5": ("Search students", search_students),
    "6": ("Add course", add_course),
    "7": ("List courses", list_courses),
    "8": ("Enroll student in course", enroll_student),
    "9": ("Record marks", record_marks),
    "10": ("View transcript", transcript),
    "0": ("Exit", None),
}


def main():
    conn = connect()
    create_tables(conn)
    while True:
        print("\n=== Student Record Management System ===")
        for key, (label, _) in MENU.items():
            print(f"{key:>2}. {label}")
        choice = input("Choose an option: ").strip()
        if choice == "0":
            print("Goodbye.")
            break
        if choice in MENU:
            MENU[choice][1](conn)
        else:
            print("Invalid option.")
    conn.close()


main()


=== Student Record Management System ===
 1. Add student
 2. List students
 3. Update student
 4. Delete student
 5. Search students
 6. Add course
 7. List courses
 8. Enroll student in course
 9. Record marks
10. View transcript
 0. Exit


In [ ]:
%%writefile student_records.py
import sqlite3
import sys

DB_FILE = "student_records.db"

def get_db_connection():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def init_db():
    with get_db_connection() as conn:
        cursor = conn.cursor()

        # Create Students Table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS students (
            roll_no TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL
        );
        """)

        # Create Courses Table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS courses (
            course_code TEXT PRIMARY KEY,
            course_name TEXT NOT NULL,
            credits INTEGER NOT NULL CHECK (credits > 0)
        );
        """)

        # Create Enrollments Table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS enrollments (
            roll_no TEXT,
            course_code TEXT,
            marks INTEGER CHECK (marks >= 0 AND marks <= 100),
            PRIMARY KEY (roll_no, course_code),
            FOREIGN KEY (roll_no) REFERENCES students(roll_no) ON DELETE CASCADE,
            FOREIGN KEY (course_code) REFERENCES courses(course_code) ON DELETE CASCADE
        );
        """)
        conn.commit()

def get_letter_grade(marks):
    if marks is None: return "N/A"
    if 90 <= marks <= 100: return "A+"
    if 80 <= marks < 90: return "A"
    if 70 <= marks < 80: return "B"
    if 60 <= marks < 70: return "C"
    if 50 <= marks < 60: return "D"
    return "F"

# --- Student Methods ---
def add_student(roll_no, name, email):
    try:
        with get_db_connection() as conn:
            conn.execute("INSERT INTO students VALUES (?, ?, ?)", (roll_no, name, email))
            print(f"Student {name} added successfully!")
    except sqlite3.IntegrityError as e:
        print(f"Database Error: {e}. Check if roll number or email is already registered.")

def view_students():
    with get_db_connection() as conn:
        students = conn.execute("SELECT * FROM students").fetchall()
        if not students:
            print("No students found.")
            return
        print("\n--- Students ---")
        for r, n, e in students:
            print(f"Roll No: {r} | Name: {n} | Email: {e}")

# --- Course Methods ---
def add_course(code, name, credits):
    try:
        with get_db_connection() as conn:
            conn.execute("INSERT INTO courses VALUES (?, ?, ?)", (code, name, credits))
            print(f"Course {name} ({code}) added successfully!")
    except sqlite3.IntegrityError as e:
        print(f"Database Error: {e}")

# --- Enrollment Methods ---
def enroll_student(roll_no, course_code):
    try:
        with get_db_connection() as conn:
            conn.execute("INSERT INTO enrollments (roll_no, course_code) VALUES (?, ?)", (roll_no, course_code))
            print(f"Enrolled student {roll_no} in course {course_code} successfully!")
    except sqlite3.IntegrityError as e:
        print(f"Database Error: {e}. Verify student and course exist and are not already connected.")

def assign_marks(roll_no, course_code, marks):
    try:
        with get_db_connection() as conn:
            cur = conn.cursor()
            cur.execute("UPDATE enrollments SET marks = ? WHERE roll_no = ? AND course_code = ?", (marks, roll_no, course_code))
            if cur.rowcount == 0:
                print("No matching enrollment found to update.")
            else:
                print(f"Marks updated to {marks} ({get_letter_grade(marks)})!")
    except sqlite3.IntegrityError as e:
        print(f"Database Error: {e}. Marks must be between 0 and 100.")

# --- Reports ---
def generate_transcript(roll_no):
    with get_db_connection() as conn:
        student = conn.execute("SELECT name FROM students WHERE roll_no = ?", (roll_no,)).fetchone()
        if not student:
            print("Student not found.")
            return

        query = """
            SELECT e.course_code, c.course_name, c.credits, e.marks
            FROM enrollments e
            JOIN courses c ON e.course_code = c.course_code
            WHERE e.roll_no = ?
        """
        records = conn.execute(query, (roll_no,)).fetchall()

        print(f"\n================ TRANSCRIPT FOR {student[0].upper()} ({roll_no}) ================")
        if not records:
            print("No enrolled courses found for this student.")
            return

        total_weighted_marks = 0
        total_credits = 0

        print(f"{'Course Code':<15} | {'Course Name':<25} | {'Credits':<8} | {'Marks':<5} | {'Grade':<5}")
        print("-" * 70)
        for code, name, credits, marks in records:
            grade = get_letter_grade(marks)
            marks_str = str(marks) if marks is not None else "N/A"
            print(f"{code:<15} | {name:<25} | {credits:<8} | {marks_str:<5} | {grade:<5}")

            if marks is not None:
                total_weighted_marks += marks * credits
                total_credits += credits

        print("-" * 70)
        if total_credits > 0:
            weighted_average = total_weighted_marks / total_credits
            print(f"Credit-Weighted Average Marks: {weighted_average:.2f} ({get_letter_grade(weighted_average)})")
        else:
            print("No grades recorded yet to calculate GPA.")

# --- Menu Loop ---
def menu():
    init_db()
    while True:
        print("\n=== Student Record System ===")
        print("1. Add Student")
        print("2. View All Students")
        print("3. Add Course")
        print("4. Enroll Student in Course")
        print("5. Enter Marks")
        print("6. Generate Transcript")
        print("7. Exit")

        choice = input("Select an option (1-7): ").strip()
        if choice == '1':
            r = input("Roll No: ").strip()
            n = input("Name: ").strip()
            e = input("Email: ").strip()
            if r and n and e: add_student(r, n, e)
        elif choice == '2':
            view_students()
        elif choice == '3':
            c = input("Course Code: ").strip()
            n = input("Course Name: ").strip()
            cr = input("Credits: ").strip()
            if c and n and cr.isdigit(): add_course(c, n, int(cr))
        elif choice == '4':
            r = input("Roll No: ").strip()
            c = input("Course Code: ").strip()
            if r and c: enroll_student(r, c)
        elif choice == '5':
            r = input("Roll No: ").strip()
            c = input("Course Code: ").strip()
            m = input("Marks (0-100): ").strip()
            if r and c and m.isdigit(): assign_marks(r, c, int(m))
        elif choice == '6':
            r = input("Roll No: ").strip()
            if r: generate_transcript(r)
        elif choice == '7':
            print("Goodbye!")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    # If arguments are passed, we could handle cli args, otherwise run interactive menu
    menu()

You can test the created module directly inside this notebook environment by importing its functions directly to register records programmatically, without needing to interact with the input menu.

In [ ]:
import student_records

# Initialize database tables
student_records.init_db()

# Add Sample Records
student_records.add_student("S101", "Alice Smith", "alice@univ.edu")
student_records.add_course("CS101", "Intro to Computer Science", 4)
student_records.add_course("MATH201", "Linear Algebra", 3)

# Enroll and Set Grades
student_records.enroll_student("S101", "CS101")
student_records.enroll_student("S101", "MATH201")

student_records.assign_marks("S101", "CS101", 95)     # Grade A+
student_records.assign_marks("S101", "MATH201", 82)   # Grade A

# Verify Transcript Calculation
student_records.generate_transcript("S101")

Database Error: UNIQUE constraint failed: students.email. Check if roll number or email is already registered.
Database Error: UNIQUE constraint failed: courses.course_code
Database Error: UNIQUE constraint failed: courses.course_code
Database Error: UNIQUE constraint failed: enrollments.roll_no, enrollments.course_code. Verify student and course exist and are not already connected.
Database Error: UNIQUE constraint failed: enrollments.roll_no, enrollments.course_code. Verify student and course exist and are not already connected.
Marks updated to 95 (A+)!
Marks updated to 82 (A)!

================ TRANSCRIPT FOR ALICE SMITH (S101) ================
Course Code     | Course Name               | Credits  | Marks | Grade
----------------------------------------------------------------------
CS101           | Intro to Computer Science | 4        | 95    | A+   
MATH201         | Linear Algebra            | 3        | 82    | A    
-----------------------------------------------------------